In [4]:
import pandas as pd
import numpy as np

cohort = pd.read_pickle(
    "../results/cohort_features_checkpoint.pkl"
)

# All categorical predictors from the verified paper map
all_categorical_cols = [
    "RIAGENDR",          # Gender
    "RIDRETH1",          # Race
    "INDHHINC",          # Income
    "CVDFITLV",          # Cardiovascular fitness level
    "PAQ180",            # Physical activity
    "BPQ020",            # Hypertension history
    "XY_Estimation",     # DNAm estimated sex
    "MCQ160B",           # Heart failure history
    "MCQ160C",           # Coronary artery disease history
    "MCQ160F",           # Cerebrovascular/stroke history
    "DERIVED_HL_MED",    # Hyperlipidemia medication
    "DERIVED_HTN_MED",   # Hypertension medication
    "DERIVED_DM_MED",    # Diabetes medication
    "DIQ010",            # Diabetes history
]

# Only use columns that currently exist
categorical_cols = [
    col for col in all_categorical_cols
    if col in cohort.columns
]

missing_categorical = [
    col for col in all_categorical_cols
    if col not in cohort.columns
]

print("Categorical columns available now:")
print(categorical_cols)

print("\nStill need to build later:")
print(missing_categorical)

Categorical columns available now:
['RIAGENDR', 'RIDRETH1', 'INDHHINC', 'CVDFITLV', 'PAQ180', 'BPQ020', 'MCQ160B', 'MCQ160C', 'MCQ160F', 'DIQ010']

Still need to build later:
['XY_Estimation', 'DERIVED_HL_MED', 'DERIVED_HTN_MED', 'DERIVED_DM_MED']


In [5]:
cohort_encoded = pd.get_dummies(
    cohort,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 303)


In [6]:
for col in categorical_cols:
    new_cols = [
        c for c in cohort_encoded.columns
        if c.startswith(col + "_")
    ]

    print(col, "->", new_cols)

RIAGENDR -> ['RIAGENDR_1.0', 'RIAGENDR_2.0']
RIDRETH1 -> ['RIDRETH1_1.0', 'RIDRETH1_2.0', 'RIDRETH1_3.0', 'RIDRETH1_4.0', 'RIDRETH1_5.0']
INDHHINC -> ['INDHHINC_1.0', 'INDHHINC_2.0', 'INDHHINC_3.0', 'INDHHINC_4.0', 'INDHHINC_5.0', 'INDHHINC_6.0', 'INDHHINC_7.0', 'INDHHINC_8.0', 'INDHHINC_9.0', 'INDHHINC_10.0', 'INDHHINC_11.0', 'INDHHINC_12.0', 'INDHHINC_13.0', 'INDHHINC_77.0', 'INDHHINC_99.0']
CVDFITLV -> ['CVDFITLV_1.0', 'CVDFITLV_2.0', 'CVDFITLV_3.0']
PAQ180 -> ['PAQ180_1.0', 'PAQ180_2.0', 'PAQ180_3.0', 'PAQ180_4.0']
BPQ020 -> ['BPQ020_1.0', 'BPQ020_2.0', 'BPQ020_9.0']
MCQ160B -> ['MCQ160B_1.0', 'MCQ160B_2.0', 'MCQ160B_7.0', 'MCQ160B_9.0']
MCQ160C -> ['MCQ160C_1.0', 'MCQ160C_2.0', 'MCQ160C_7.0', 'MCQ160C_9.0']
MCQ160F -> ['MCQ160F_1.0', 'MCQ160F_2.0', 'MCQ160F_9.0']
DIQ010 -> ['DIQ010_1.0', 'DIQ010_2.0', 'DIQ010_3.0', 'DIQ010_7.0', 'DIQ010_9.0']


In [7]:
cohort_ohe = cohort.copy()

special_missing = {
    "INDHHINC": [77, 99],
    "PAQ180": [7, 9],
    "BPQ020": [7, 9],
    "MCQ160B": [7, 9],
    "MCQ160C": [7, 9],
    "MCQ160F": [7, 9],
    "DIQ010": [7, 9],
}

for col, codes in special_missing.items():
    if col in cohort_ohe.columns:
        cohort_ohe[col] = cohort_ohe[col].replace(codes, np.nan)

In [8]:
cohort_encoded = pd.get_dummies(
    cohort_ohe,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort_ohe.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 293)


In [9]:
bad_dummy_cols = [
    c for c in cohort_encoded.columns
    if any(
        c.endswith(suffix)
        for suffix in [
            "_7.0",
            "_9.0",
            "_77.0",
            "_99.0"
        ]
    )
]

print("Invalid dummy columns:", bad_dummy_cols)

Invalid dummy columns: ['INDHHINC_7.0', 'INDHHINC_9.0']


In [10]:
[c for c in cohort.columns if "mort" in c.lower() or "death" in c.lower()]

['MORTSTAT', 'death_5yr']

In [11]:
cohort_encoded["mortality_5yr"] = (
    (cohort_encoded["MORTSTAT"] == 1) &
    (cohort_encoded["PERMTH_INT"] <= 60)
).astype(int)

print(cohort_encoded["mortality_5yr"].value_counts())

mortality_5yr
0    8664
1    1293
Name: count, dtype: int64


In [12]:
y = cohort_encoded["mortality_5yr"]

majority_class = y.mode()[0]

y_pred_stupid = np.full(
    len(y),
    majority_class
)

accuracy = (y_pred_stupid == y).mean()

print("Majority class:", majority_class)
print(f"Stupid model accuracy: {accuracy:.2%}")

Majority class: 0
Stupid model accuracy: 87.01%


In [13]:
print(
    "Targets identical:",
    (cohort_encoded["death_5yr"] ==
     cohort_encoded["mortality_5yr"]).all()
)

Targets identical: True


In [14]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

y = cohort_encoded["death_5yr"].astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(np.zeros(len(y)), y),
    start=1
):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Learn majority class ONLY from this training fold
    majority_class = y_train.mode().iloc[0]

    # Predict same class for everyone in validation fold
    y_pred = np.full(
        len(y_val),
        majority_class
    )

    accuracy = (
        y_pred == y_val.to_numpy()
    ).mean()

    fold_accuracies.append(accuracy)

    print(
        f"Fold {fold}: "
        f"{accuracy:.2%}"
    )

print(
    "\nMean 5-fold accuracy:",
    f"{np.mean(fold_accuracies):.2%}"
)

print(
    "Std:",
    f"{np.std(fold_accuracies):.4f}"
)

Fold 1: 87.00%
Fold 2: 87.00%
Fold 3: 87.04%
Fold 4: 87.04%
Fold 5: 86.99%

Mean 5-fold accuracy: 87.01%
Std: 0.0002


In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np

In [40]:
y = cohort["death_5yr"].astype(int)

# Never use outcome/linkage fields as predictors
drop_cols = [
    "SEQN",
    "death_5yr",
    "MORTSTAT",
    "PERMTH_INT"
]

X = cohort.drop(
    columns=[c for c in drop_cols if c in cohort.columns]
)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Categorical: 10
Numeric: 251


In [38]:
X_all = X.copy()

In [17]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    # Learn preprocessing ONLY from training fold
    X_train_processed = preprocessor.fit_transform(X_train)

    # Apply same learned preprocessing to validation
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 2: train = (7965, 297) | validation = (1992, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 3: train = (7966, 297) | validation = (1991, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [19]:
all_missing_cols = [
    col for col in X.columns
    if X[col].isna().all()
]

print("Completely empty columns:", len(all_missing_cols))
print(all_missing_cols)

Completely empty columns: 2
['DMDEDUC3', 'DMDSCHOL']


In [20]:
X = X.drop(columns=all_missing_cols)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Predictors remaining:", X.shape[1])
print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Predictors remaining: 259
Categorical: 10
Numeric: 249


In [21]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [22]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)
Fold 2: train = (7965, 297) | validation = (1992, 297)
Fold 3: train = (7966, 297) | validation = (1991, 297)
Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


In [23]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Load data
cohort = pd.read_pickle("../results/cohort_features_checkpoint.pkl")

# 2. Select three features
glm_features = ["RIDAGEYR", "SSTNT", "SSBNP"]

X_glm = cohort[glm_features].copy()
y_glm = cohort["death_5yr"].astype(int)

# 3. Build Logistic Regression
glm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# 4. Stratified 5-fold cross-validation
glm_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 5. Evaluate
glm_scores = cross_validate(
    glm,
    X_glm,
    y_glm,
    cv=glm_cv,
    scoring={
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC-AUC": "roc_auc"
    },
    error_score="raise"
)

# 6. Display results
glm_results = pd.DataFrame({
    "Precision": glm_scores["test_Precision"],
    "Recall": glm_scores["test_Recall"],
    "F1": glm_scores["test_F1"],
    "ROC-AUC": glm_scores["test_ROC-AUC"]
})

glm_results.index = range(1, 6)

print("Logistic Regression - 5-Fold Results")
display(glm_results.round(3))

print("\nAverage Performance:")
display(glm_results.mean().round(3))

Logistic Regression - 5-Fold Results


,Precision,Recall,F1,ROC-AUC
1,0.282,0.741,0.409,0.792
2,0.292,0.749,0.420,0.802
3,0.291,0.764,0.421,0.821
4,0.325,0.829,0.467,0.852
5,0.288,0.772,0.420,0.814



Average Performance:


Precision    0.296
Recall       0.771
F1           0.427
ROC-AUC      0.816
dtype: float64

## Random Forest Model

Compare Random Forest with Logistic Regression using
the same three features and stratified 5-fold cross-validation.

In [24]:
from sklearn.ensemble import RandomForestClassifier

# 1. Build Random Forest
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Evaluate using the same 5 folds
rf_scores = cross_validate(
    rf,
    X_glm,
    y_glm,
    cv=glm_cv,
    scoring={
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC-AUC": "roc_auc"
    },
    error_score="raise"
)

# 3. Display Random Forest results
rf_results = pd.DataFrame({
    "Precision": rf_scores["test_Precision"],
    "Recall": rf_scores["test_Recall"],
    "F1": rf_scores["test_F1"],
    "ROC-AUC": rf_scores["test_ROC-AUC"]
})

rf_results.index = range(1, 6)

print("Random Forest - 5-Fold Results")
display(rf_results.round(3))

print("\nAverage Performance:")
display(rf_results.mean().round(3))

# 4. Compare both models
comparison = pd.DataFrame({
    "Logistic Regression": glm_results.mean(),
    "Random Forest": rf_results.mean()
}).T

print("\nModel Comparison:")
display(comparison.round(3))

Random Forest - 5-Fold Results


,Precision,Recall,F1,ROC-AUC
1,0.331,0.629,0.434,0.808
2,0.338,0.695,0.455,0.817
3,0.342,0.729,0.465,0.834
4,0.393,0.771,0.521,0.864
5,0.349,0.710,0.468,0.843



Average Performance:


Precision    0.351
Recall       0.707
F1           0.469
ROC-AUC      0.833
dtype: float64


Model Comparison:


,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.296,0.771,0.427,0.816
Random Forest,0.351,0.707,0.469,0.833


## Comparing Predictions for the Same Participants

In [25]:
import numpy as np
from sklearn.model_selection import cross_val_predict

# Predict probabilities using the same 5 folds
glm_prob = cross_val_predict(
    glm,
    X_glm,
    y_glm,
    cv=glm_cv,
    method="predict_proba"
)[:, 1]

rf_prob = cross_val_predict(
    rf,
    X_glm,
    y_glm,
    cv=glm_cv,
    method="predict_proba"
)[:, 1]

# Compare predictions for every participant
prediction_comparison = X_glm.copy()

prediction_comparison["Actual_Death"] = y_glm.to_numpy()
prediction_comparison["GLM_Probability"] = glm_prob
prediction_comparison["RF_Probability"] = rf_prob

# Absolute difference between the two models
prediction_comparison["Difference"] = np.abs(
    glm_prob - rf_prob
)

# Show participants with the largest differences
print("Participants with largest model disagreement:")

display(
    prediction_comparison
    .sort_values("Difference", ascending=False)
    .head(10)
    .round(3)
)

print(
    "Average probability difference:",
    round(prediction_comparison["Difference"].mean(), 3)
)

Participants with largest model disagreement:


,RIDAGEYR,SSTNT,SSBNP,Actual_Death,GLM_Probability,RF_Probability,Difference
5171,84.0,6.13,57.03,0,0.744,0.112,0.631
9276,82.0,8.81,106.30,0,0.731,0.109,0.622
3009,53.0,143.80,12.30,0,0.966,0.367,0.599
6394,82.0,8.80,158.80,0,0.731,0.134,0.597
180,82.0,8.16,145.80,0,0.734,0.137,0.596
6038,80.0,6.86,49.56,0,0.682,0.090,0.591
1215,79.0,6.34,95.55,1,0.670,0.095,0.575
1998,82.0,7.94,169.00,0,0.722,0.147,0.575
1938,81.0,5.94,49.99,0,0.701,0.127,0.574
6372,82.0,7.47,157.70,0,0.731,0.174,0.557


Average probability difference: 0.148


In [26]:
# Compare yes/no predictions using a 0.5 threshold
prediction_comparison["GLM_Predicted"] = (
    prediction_comparison["GLM_Probability"] >= 0.5
).astype(int)

prediction_comparison["RF_Predicted"] = (
    prediction_comparison["RF_Probability"] >= 0.5
).astype(int)

# Check how often the models disagree
disagree = (
    prediction_comparison["GLM_Predicted"]
    != prediction_comparison["RF_Predicted"]
)

print("Total participants:", len(prediction_comparison))
print("Different classifications:", disagree.sum())
print("Disagreement rate:", round(disagree.mean() * 100, 2), "%")

Total participants: 9957
Different classifications: 1633
Disagreement rate: 16.4 %


In [27]:
# Compare the two models' yes/no predictions
agreement_table = pd.crosstab(
    prediction_comparison["GLM_Predicted"],
    prediction_comparison["RF_Predicted"],
    rownames=["Logistic Regression"],
    colnames=["Random Forest"]
)

print("Model Agreement Table:")
print("0 = Predicts no death")
print("1 = Predicts death")

display(agreement_table)

Model Agreement Table:
0 = Predicts no death
1 = Predicts death


Random Forest,0,1
Logistic Regression,,
0,6149,434
1,1199,2175


## Comparing Individual Feature Explanations

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# Same training and test participants for both models
X_train, X_test, y_train, y_test = train_test_split(
    X_glm,
    y_glm,
    test_size=0.20,
    stratify=y_glm,
    random_state=42
)

# Fit separate copies of our existing models
glm_explain = clone(glm)
rf_explain = clone(rf)

glm_explain.fit(X_train, y_train)
rf_explain.fit(X_train, y_train)

print("Training participants:", len(X_train))
print("Test participants:", len(X_test))
print("Features:", X_train.columns.tolist())
print("Both models are ready for explanation.")

Training participants: 7965
Test participants: 1992
Features: ['RIDAGEYR', 'SSTNT', 'SSBNP']
Both models are ready for explanation.


In [29]:
%pip install "shap>=0.54.1"


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import shap
import numpy as np
import pandas as pd

# Use the same participants for both models
background = X_train.sample(n=40, random_state=42)
people = X_test.sample(n=20, random_state=42)

# Return predicted probability of death
def glm_probability(data):
    data = pd.DataFrame(data, columns=glm_features)
    return glm_explain.predict_proba(data)[:, 1]

def rf_probability(data):
    data = pd.DataFrame(data, columns=glm_features)
    return rf_explain.predict_proba(data)[:, 1]

# Use the same SHAP method and background for both models
glm_explainer = shap.Explainer(
    glm_probability, background,
    algorithm="permutation", seed=42
)

rf_explainer = shap.Explainer(
    rf_probability, background,
    algorithm="permutation", seed=42
)

# Explain the same 20 held-out participants
glm_shap = glm_explainer(people, max_evals=35)
rf_shap = rf_explainer(people, max_evals=35)

# Find the most influential feature for each participant
glm_top = np.abs(glm_shap.values).argmax(axis=1)
rf_top = np.abs(rf_shap.values).argmax(axis=1)

explanation_comparison = pd.DataFrame({
    "GLM_Top_Feature": [glm_features[i] for i in glm_top],
    "RF_Top_Feature": [glm_features[i] for i in rf_top],
    "Agree": glm_top == rf_top
}, index=people.index)

print("Feature importance comparison:")
display(explanation_comparison)

print(
    "\nTop-feature agreement:",
    round(explanation_comparison["Agree"].mean() * 100, 1),
    "%"
)

Feature importance comparison:


,GLM_Top_Feature,RF_Top_Feature,Agree
1721,RIDAGEYR,SSTNT,False
3385,RIDAGEYR,SSBNP,False
2825,SSTNT,SSBNP,False
6341,RIDAGEYR,SSTNT,False
7290,SSTNT,SSTNT,True
598,RIDAGEYR,RIDAGEYR,True
7668,RIDAGEYR,SSBNP,False
3237,RIDAGEYR,RIDAGEYR,True
6252,RIDAGEYR,RIDAGEYR,True
2116,RIDAGEYR,RIDAGEYR,True



Top-feature agreement: 45.0 %


In [31]:
### Examining One Participant's SHAP Explanations

In [32]:
# Select one participant where the models disagree
person_id = explanation_comparison.index[
    ~explanation_comparison["Agree"]
][0]

position = people.index.get_loc(person_id)

# Compare the SHAP contributions for each feature
person_explanation = pd.DataFrame({
    "Feature": glm_features,
    "Actual_Value": people.loc[person_id, glm_features].values,
    "GLM_SHAP": glm_shap.values[position],
    "RF_SHAP": rf_shap.values[position]
})

print("Participant index:", person_id)

display(person_explanation.round(4))

# Compare model predictions for the same person
person = people.loc[[person_id]]

print(
    "Logistic Regression prediction:",
    round(glm_explain.predict_proba(person)[0, 1], 3)
)

print(
    "Random Forest prediction:",
    round(rf_explain.predict_proba(person)[0, 1], 3)
)


Participant index: 1721


,Feature,Actual_Value,GLM_SHAP,RF_SHAP
0,RIDAGEYR,60.00,-0.0934,-0.0775
1,SSTNT,5.39,-0.0300,-0.1136
2,SSBNP,11.44,-0.0065,-0.0306


Logistic Regression prediction: 0.293
Random Forest prediction: 0.121


In [33]:
# Test SHAP agreement on 100 held-out participants
people_100 = X_test.sample(n=100, random_state=123)

# Explain both models on exactly the same people
glm_shap_100 = glm_explainer(people_100, max_evals=35)
rf_shap_100 = rf_explainer(people_100, max_evals=35)

# Most influential feature for each person
glm_top_100 = np.abs(glm_shap_100.values).argmax(axis=1)
rf_top_100 = np.abs(rf_shap_100.values).argmax(axis=1)

# Measure agreement
agree_100 = glm_top_100 == rf_top_100

print("Participants tested:", len(people_100))
print("Models agree:", agree_100.sum())
print("Models disagree:", (~agree_100).sum())
print("Agreement rate:", round(agree_100.mean() * 100, 1), "%")
print("Disagreement rate:", round((~agree_100).mean() * 100, 1), "%")

PermutationExplainer explainer: 101it [00:27,  2.42it/s]                         

Participants tested: 100
Models agree: 49
Models disagree: 51
Agreement rate: 49.0 %
Disagreement rate: 51.0 %


In [34]:
# Compare which features each model ranks first

glm_top_features = np.array(glm_features)[glm_top_100]
rf_top_features = np.array(glm_features)[rf_top_100]

feature_agreement_table = pd.crosstab(
    pd.Series(glm_top_features, name="Logistic Regression"),
    pd.Series(rf_top_features, name="Random Forest")
)

print("Top Feature Agreement Table:")
display(feature_agreement_table)

print("\nMost common top feature for Logistic Regression:")
print(pd.Series(glm_top_features).value_counts())

print("\nMost common top feature for Random Forest:")
print(pd.Series(rf_top_features).value_counts())

Top Feature Agreement Table:


Random Forest,RIDAGEYR,SSBNP,SSTNT
Logistic Regression,,,
RIDAGEYR,47,16,34
SSTNT,0,1,2



Most common top feature for Logistic Regression:
RIDAGEYR    97
SSTNT        3
Name: count, dtype: int64

Most common top feature for Random Forest:
RIDAGEYR    47
SSTNT       36
SSBNP       17
Name: count, dtype: int64


### Preliminary Findings

We compared SHAP explanations from Logistic Regression and Random Forest
for 100 held-out NHANES participants.

The models agreed on the most important feature for 49% of participants
and disagreed for 51%.

Logistic Regression mostly ranked age first (97/100), while Random
Forest distributed its top features across age, Troponin-T, and NT-proBNP.

These findings suggest that models with similar predictive performance
may provide different explanations for the same person.

This is a preliminary analysis using three features and 5-year
all-cause mortality.

### Comparing Complete SHAP Feature Rankings

Compare how Logistic Regression and Random Forest rank
all three features for the same 100 participants.

In [35]:
from scipy.stats import spearmanr

# Rank features from most to least influential
glm_ranks = np.argsort(-np.abs(glm_shap_100.values), axis=1)
rf_ranks = np.argsort(-np.abs(rf_shap_100.values), axis=1)

# Do the complete rankings match?
full_agreement = np.all(glm_ranks == rf_ranks, axis=1)

# Measure similarity between the full rankings
rank_correlations = [
    spearmanr(
        np.abs(glm_shap_100.values[i]),
        np.abs(rf_shap_100.values[i])
    ).statistic
    for i in range(len(people_100))
]

# Create a comparison table
ranking_comparison = pd.DataFrame({
    "GLM_Ranking": [
        " > ".join(np.array(glm_features)[r])
        for r in glm_ranks
    ],
    "RF_Ranking": [
        " > ".join(np.array(glm_features)[r])
        for r in rf_ranks
    ],
    "Full_Agreement": full_agreement,
    "Spearman": rank_correlations
}, index=people_100.index)

print("Participants:", len(people_100))
print("Complete ranking matches:", full_agreement.sum())
print("Complete ranking agreement:",
      round(full_agreement.mean() * 100, 1), "%")

print("Average rank correlation:",
      round(np.nanmean(rank_correlations), 3))

display(ranking_comparison.head(10))


Participants: 100
Complete ranking matches: 23
Complete ranking agreement: 23.0 %
Average rank correlation: 0.265


,GLM_Ranking,RF_Ranking,Full_Agreement,Spearman
4764,RIDAGEYR > SSTNT > SSBNP,SSBNP > RIDAGEYR > SSTNT,False,-0.5
3217,RIDAGEYR > SSTNT > SSBNP,RIDAGEYR > SSTNT > SSBNP,True,1.0
5260,RIDAGEYR > SSTNT > SSBNP,RIDAGEYR > SSTNT > SSBNP,True,1.0
3179,RIDAGEYR > SSTNT > SSBNP,RIDAGEYR > SSBNP > SSTNT,False,0.5
2171,RIDAGEYR > SSTNT > SSBNP,RIDAGEYR > SSTNT > SSBNP,True,1.0
9223,RIDAGEYR > SSTNT > SSBNP,SSTNT > RIDAGEYR > SSBNP,False,0.5
4924,RIDAGEYR > SSTNT > SSBNP,SSBNP > SSTNT > RIDAGEYR,False,-1.0
3759,RIDAGEYR > SSTNT > SSBNP,SSTNT > SSBNP > RIDAGEYR,False,-0.5
1930,RIDAGEYR > SSBNP > SSTNT,SSBNP > RIDAGEYR > SSTNT,False,0.5
5298,RIDAGEYR > SSTNT > SSBNP,SSTNT > SSBNP > RIDAGEYR,False,-0.5


### Checking Whether Missing Biomarkers Affect Agreement

In [36]:
# Check missing biomarker values in our 100 participants
missing_check = pd.DataFrame(index=people_100.index)

missing_check["Missing_Biomarker"] = people_100[
    ["SSTNT", "SSBNP"]
].isna().any(axis=1)

missing_check["Top_Feature_Agree"] = agree_100
missing_check["Full_Ranking_Agree"] = full_agreement

# Compare agreement for participants with and without missing data
missing_results = missing_check.groupby(
    "Missing_Biomarker"
).agg(
    Participants=("Full_Ranking_Agree", "size"),
    Top_Agreement=("Top_Feature_Agree", "mean"),
    Full_Agreement=("Full_Ranking_Agree", "mean")
)

print("Agreement by biomarker availability:")
display(missing_results.round(3))

print("\nIndividual biomarker missing counts:")
print(people_100[["SSTNT", "SSBNP"]].isna().sum())

Agreement by biomarker availability:


,Participants,Top_Agreement,Full_Agreement
Missing_Biomarker,,,
False,84,0.429,0.274
True,16,0.812,0.000



Individual biomarker missing counts:
SSTNT    16
SSBNP    16
dtype: int64


### Preliminary Research Findings

We compared Logistic Regression and Random Forest explanations
for 100 held-out NHANES participants.

- Top-feature agreement: 49%
- Complete-ranking agreement: 23%
- Average Spearman correlation: 0.265

We also examined missing biomarker data. Among the 84 participants
with complete biomarker measurements, top-feature agreement was
42.9%, compared with 81.2% among the 16 participants with missing
measurements.

These are preliminary findings based on three features and
5-year all-cause mortality. Further validation is needed.

### Larger SHAP Experiment: Complete Biomarker Data

Compare Logistic Regression and Random Forest explanations
for 300 held-out participants with both biomarker measurements.

Measure top-feature agreement, complete ranking agreement,
and average Spearman correlation.

In [37]:
from scipy.stats import spearmanr

# 1. Keep participants with both biomarkers measured
complete_people = X_test.dropna(
    subset=["SSTNT", "SSBNP"]
)

# 2. Select 300 held-out participants
people_300 = complete_people.sample(
    n=min(300, len(complete_people)),
    random_state=123
)

# 3. Calculate SHAP explanations
glm_shap_300 = glm_explainer(
    people_300,
    max_evals=35
)

rf_shap_300 = rf_explainer(
    people_300,
    max_evals=35
)

# 4. Rank features by absolute SHAP values
glm_ranks_300 = np.argsort(
    -np.abs(glm_shap_300.values),
    axis=1
)

rf_ranks_300 = np.argsort(
    -np.abs(rf_shap_300.values),
    axis=1
)

# 5. Compare most important feature
top_agree_300 = (
    glm_ranks_300[:, 0] == rf_ranks_300[:, 0]
)

# 6. Compare complete feature rankings
full_agree_300 = np.all(
    glm_ranks_300 == rf_ranks_300,
    axis=1
)

# 7. Calculate Spearman correlations
spearman_300 = [
    spearmanr(
        np.abs(glm_shap_300.values[i]),
        np.abs(rf_shap_300.values[i])
    ).statistic
    for i in range(len(people_300))
]

# 8. Display results
print("Complete-biomarker SHAP Experiment")
print("--------------------------------")
print("Participants:", len(people_300))

print(
    "Top-feature agreement:",
    round(top_agree_300.mean() * 100, 1), "%"
)

print(
    "Complete ranking agreement:",
    round(full_agree_300.mean() * 100, 1), "%"
)

print(
    "Average Spearman correlation:",
    round(np.nanmean(spearman_300), 3)
)

PermutationExplainer explainer:  83%|████████▎ | 249/300 [01:08<00:16,  3.02it/s]


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np

# Baseline always predicts no death (class 0)
baseline_predictions = np.zeros(len(y_glm), dtype=int)

# Constant scores for ROC-AUC
baseline_probabilities = np.zeros(len(y_glm))

print("Stupid Baseline Model")
print("---------------------")

print("Accuracy:",
      round(accuracy_score(y_glm, baseline_predictions), 3))

print("Precision:",
      round(precision_score(
          y_glm, baseline_predictions, zero_division=0
      ), 3))

print("Recall:",
      round(recall_score(
          y_glm, baseline_predictions, zero_division=0
      ), 3))

print("F1-score:",
      round(f1_score(
          y_glm, baseline_predictions, zero_division=0
      ), 3))

print("ROC-AUC:",
      round(roc_auc_score(
          y_glm, baseline_probabilities
      ), 3))

Stupid Baseline Model
---------------------
Accuracy: 0.87
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.5


### Logistic Regression Equation and Coefficients

In [ ]:
import numpy as np
import pandas as pd

# Get our already-trained Logistic Regression model
model = glm_explain.named_steps["model"]
scaler = glm_explain.named_steps["scaler"]

# Coefficients for standardized features
scaled_coefs = model.coef_[0]

# Convert coefficients back to original feature units
raw_coefs = scaled_coefs / scaler.scale_

# Adjust the intercept
raw_intercept = model.intercept_[0] - np.sum(
    raw_coefs * scaler.mean_
)

# Display results
coefficients = pd.DataFrame({
    "Feature": glm_features,
    "Coefficient": raw_coefs
})

print("Logistic Regression Coefficients:")
display(coefficients.round(6))

print("\nIntercept (A):", round(raw_intercept, 6))

print("\nLogistic Regression Equation:")
print(
    f"z = {raw_intercept:.6f}"
    f" + ({raw_coefs[0]:.6f} * Age)"
    f" + ({raw_coefs[1]:.6f} * Troponin_T)"
    f" + ({raw_coefs[2]:.6f} * NT_proBNP)"
)

print("\nP(death within 5 years) = 1 / (1 + exp(-z))")

Logistic Regression Coefficients:


,Feature,Coefficient
0,RIDAGEYR,0.081101
1,SSTNT,0.026090
2,SSBNP,0.000182



Intercept (A): -5.889396

Logistic Regression Equation:
z = -5.889396 + (0.081101 * Age) + (0.026090 * Troponin_T) + (0.000182 * NT_proBNP)

P(death within 5 years) = 1 / (1 + exp(-z))


## Summary of Model Results

We trained Logistic Regression and Random Forest models using
age, Troponin-T, and NT-proBNP to predict 5-year all-cause mortality.

### Model Performance

| Model | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|
| Stupid Baseline | 0.000 | 0.000 | 0.000 | 0.500 |
| Logistic Regression | 0.296 | 0.771 | 0.427 | 0.816 |
| Random Forest | 0.351 | 0.707 | 0.469 | 0.833 |

Random Forest achieved higher precision, F1-score, and ROC-AUC,
while Logistic Regression achieved higher recall.

### Preliminary SHAP Findings

We compared feature explanations from both models for 300
held-out participants with complete biomarker measurements.

- Top-feature agreement: 45.3%
- Complete ranking agreement: 28.7%
- Average Spearman correlation: 0.338

These preliminary results suggest that models with similar
predictive performance may provide different explanations
for the same participants.

### Limitations

The current models use only three features and predict 5-year
all-cause mortality, not cardiovascular-specific mortality.

Further experiments are needed to test explanation stability
across different training samples.

# Cardiovascular Mortality Outcome

Identify and validate cardiovascular mortality outcomes
using the NHANES linked mortality data.

In [ ]:
import pandas as pd

# Load our existing NHANES dataset
cohort = pd.read_pickle(
    "../results/cohort_features_checkpoint.pkl"
)

print("Dataset shape:", cohort.shape)

# Find available mortality-related columns
mortality_cols = [
    col for col in cohort.columns
    if any(keyword in col.upper() for keyword in [
        "MORT", "UCOD", "CAUSE", "PERMTH", "DEATH", "SEQN"
    ])
]

print("\nMortality-related columns:")
print(mortality_cols)

# Inspect the key mortality variables
for col in ["MORTSTAT", "PERMTH_INT", "UCOD_LEADING"]:
    if col in cohort.columns:
        print(f"\n{col}:")
        print(cohort[col].value_counts(dropna=False).head(15))
    else:
        print(f"\n{col}: NOT FOUND")

Dataset shape: (9957, 265)

Mortality-related columns:
['SEQN', 'MORTSTAT', 'PERMTH_INT', 'death_5yr']

MORTSTAT:
MORTSTAT
0    5467
1    4490
Name: count, dtype: int64

PERMTH_INT:
PERMTH_INT
203    159
212    139
194    131
191    130
219    129
196    125
220    123
246    122
211    121
226    121
218    120
188    117
200    116
199    115
193    112
Name: count, dtype: int64

UCOD_LEADING: NOT FOUND


## Full-Feature Logistic Regression and Random Forest

Expand both models beyond the initial three predictors.

Use the available NHANES features while excluding participant
identifiers, mortality outcomes, and follow-up information.

In [41]:
import pandas as pd
import numpy as np

# Start with our previously prepared feature dataset
X_all = X.copy()
y_all = cohort.loc[X_all.index, "death_5yr"].astype(int)

# Exclude outcome-related information to prevent leakage
excluded_terms = [
    "SEQN", "MORT", "DEATH", "PERMTH",
    "UCOD", "ELIGSTAT", "CENSOR",
    "FOLLOWUP", "FOLLOW_UP", "SURVIVAL",
    "CAUSE"
]

excluded_cols = [
    c for c in X_all.columns
    if any(term in c.upper() for term in excluded_terms)
]

X_all = X_all.drop(columns=excluded_cols)

# Drop features with no observed values
empty_cols = X_all.columns[X_all.isna().all()].tolist()
X_all = X_all.drop(columns=empty_cols)

# Preserve the categorical variables identified earlier
categorical_all = [
    c for c in all_categorical_cols
    if c in X_all.columns
]

numeric_all = [
    c for c in X_all.columns
    if c not in categorical_all
]

# Verify numeric feature types
non_numeric = [
    c for c in numeric_all
    if not pd.api.types.is_numeric_dtype(X_all[c])
]

assert not non_numeric, f"Review nonnumeric columns: {non_numeric}"
assert X_all.index.equals(y_all.index)

print("Participants:", len(X_all))
print("Total predictors:", X_all.shape[1])
print("Numeric predictors:", len(numeric_all))
print("Categorical predictors:", len(categorical_all))

print("\nExcluded outcome-related columns:", excluded_cols)
print("Removed entirely missing columns:", empty_cols)

print("\nPredictors being used:")
print(X_all.columns.tolist())

Participants: 9957
Total predictors: 258
Numeric predictors: 248
Categorical predictors: 10

Excluded outcome-related columns: ['ELIGSTAT']
Removed entirely missing columns: ['DMDEDUC3', 'DMDSCHOL']

Predictors being used:
['SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'RIDRETH2', 'DMQMILIT', 'DMDBORN', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC2', 'DMDEDUC', 'DMDMARTL', 'DMDHHSIZ', 'INDHHINC', 'INDFMINC', 'INDFMPIR', 'RIDEXPRG', 'RIDPREG', 'DMDHRGND', 'DMDHRAGE', 'DMDHRBRN', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'WTINT2YR', 'WTINT4YR', 'WTMEC2YR', 'WTMEC4YR', 'SDMVPSU', 'SDMVSTRA', 'SDJ1REPN', 'DMAETHN', 'DMARACE', 'WTMREP01', 'WTMREP02', 'WTMREP03', 'WTMREP04', 'WTMREP05', 'WTMREP06', 'WTMREP07', 'WTMREP08', 'WTMREP09', 'WTMREP10', 'WTMREP11', 'WTMREP12', 'WTMREP13', 'WTMREP14', 'WTMREP15', 'WTMREP16', 'WTMREP17', 'WTMREP18', 'WTMREP19', 'WTMREP20', 'WTMREP21', 'WTMREP22', 'WTMREP23', 'WTMREP24', 'WTMREP25', 'WTMREP26', 'WTMREP27', 'WTMREP28', 'W

In [42]:
check_features = [
    "RIAGENDR", "RIDRETH1", "RIDRETH2",
    "DMDEDUC2", "DMDMARTL", "INDHHINC",
    "PAQ180", "BPQ020", "MCQ160B", "DIQ010"
]

checks = []

for feature in check_features:
    if feature in X_all.columns:
        checks.append({
            "Feature": feature,
            "Currently Categorical": feature in categorical_all,
            "Unique Values": X_all[feature].nunique(),
            "Example Values": str(
                X_all[feature].dropna().unique()[:8]
            )
        })

display(pd.DataFrame(checks))

,Feature,Currently Categorical,Unique Values,Example Values
0,RIAGENDR,True,2,[1. 2.]
1,RIDRETH1,True,5,[3. 4. 1. 2. 5.]
2,RIDRETH2,False,5,[1. 2. 3. 5. 4.]
3,DMDEDUC2,False,7,[5. 2. 3. 1. 4. 9. 7.]
4,DMDMARTL,False,8,[ 1. 4. 3. 2. 5. 6. 77. 99.]
5,INDHHINC,True,15,[ 8. 11. 3. 5. 1. 6. 99. 12.]
6,PAQ180,True,4,[1. 3. 2. 4.]
7,BPQ020,True,3,[2. 1. 9.]
8,MCQ160B,True,4,[2. 1. 9. 7.]
9,DIQ010,True,5,[2. 1. 3. 9. 7.]


In [43]:
# Correct categorical feature types
extra_categorical = [
    "RIDRETH2",
    "DMDEDUC2",
    "DMDMARTL"
]

categorical_all = list(dict.fromkeys(
    categorical_all +
    [c for c in extra_categorical if c in X_all.columns]
))

numeric_all = [
    c for c in X_all.columns
    if c not in categorical_all
]

# Clean special missing-response codes
missing_codes = {
    "DMDEDUC2": [7, 9],
    "DMDMARTL": [77, 99],
    "INDHHINC": [77, 99],
    "BPQ020": [7, 9],
    "MCQ160B": [7, 9],
    "DIQ010": [7, 9]
}

for col, codes in missing_codes.items():
    if col in X_all.columns:
        X_all[col] = X_all[col].replace(codes, np.nan)

print("Total predictors:", X_all.shape[1])
print("Categorical:", len(categorical_all))
print("Numeric:", len(numeric_all))
print("\nCorrected categorical variables:")
print(extra_categorical)

Total predictors: 258
Categorical: 13
Numeric: 245

Corrected categorical variables:
['RIDRETH2', 'DMDEDUC2', 'DMDMARTL']


In [44]:
# Find additional demographic categories to review
review_cols = [
    c for c in X_all.columns
    if c.startswith(("RID", "DMD", "DMQ", "IND"))
    and c not in categorical_all
    and X_all[c].nunique(dropna=True) <= 20
]

review = pd.DataFrame({
    "Feature": review_cols,
    "Unique Values": [
        X_all[c].nunique(dropna=True)
        for c in review_cols
    ],
    "Examples": [
        str(X_all[c].dropna().unique()[:8])
        for c in review_cols
    ]
})

display(review)

,Feature,Unique Values,Examples
0,RIDSTATR,2,[2. 1.]
1,RIDEXMON,2,[2. 1.]
2,DMQMILIT,4,[1. 2. 9. 7.]
3,DMDBORN,4,[1. 2. 3. 7.]
4,DMDCITZN,4,[1. 2. 9. 7.]
5,DMDYRSUS,12,[ 6. 8. 7. 9. 5. 99. 4. 2.]
6,DMDEDUC,5,[3. 1. 2. 9. 7.]
7,DMDHHSIZ,7,[1. 3. 2. 5. 4. 6. 7.]
8,INDFMINC,15,[ 8. 11. 99. 3. 5. 1. 6. 7.]
9,RIDEXPRG,3,[2. 3. 1.]


In [48]:
# 1. Remove survey administration and design fields
exclude_admin = [
    "RIDSTATR", "RIDEXMON", "SDDSRVYR",
    "SDMVPSU", "SDMVSTRA", "SDJ1REPN",
    "DMAETHN", "DMARACE"
]

remove_cols = [
    c for c in X_all.columns
    if c in exclude_admin or c.startswith("WT")
]

X_all = X_all.drop(columns=remove_cols)

# 2. Correct additional categorical features
extra_categories = [
    "DMQMILIT", "DMDBORN", "DMDCITZN",
    "DMDYRSUS", "DMDEDUC", "INDFMINC",
    "RIDEXPRG", "RIDPREG",
    "DMDHRGND", "DMDHRBRN", "DMDHREDU",
    "DMDHRMAR", "DMDHSEDU"
]

categorical_all = list(dict.fromkeys(
    [c for c in categorical_all if c in X_all.columns]
    + [c for c in extra_categories if c in X_all.columns]
))

# 3. Clean documented missing-response codes
missing_codes = {
    "DMQMILIT": [7, 9],
    "DMDBORN": [7, 9],
    "DMDCITZN": [7, 9],
    "DMDYRSUS": [77, 88, 99],
    "DMDEDUC": [7, 9],
    "INDFMINC": [77, 99],
    "DMDHRBRN": [7, 9],
    "DMDHREDU": [7, 9],
    "DMDHRMAR": [77, 99],
    "DMDHSEDU": [7, 9],
}

for col, codes in missing_codes.items():
    if col in X_all.columns:
        X_all[col] = X_all[col].replace(codes, np.nan)

# 4. Rebuild numeric feature list
numeric_all = [
    c for c in X_all.columns
    if c not in categorical_all
]

print("Participants:", len(X_all))
print("Total predictors:", X_all.shape[1])
print("Categorical predictors:", len(categorical_all))
print("Numeric predictors:", len(numeric_all))
print("Excluded administrative columns:", remove_cols)

assert X_all.index.equals(y_all.index)
assert set(categorical_all).isdisjoint(numeric_all)

Participants: 9957
Total predictors: 142
Categorical predictors: 26
Numeric predictors: 116
Excluded administrative columns: []


In [49]:
# Compare original predictors with our cleaned dataset

removed_features = [
    c for c in X.columns
    if c not in X_all.columns
]

weight_features = [
    c for c in removed_features
    if c.startswith("WT")
]

other_removed = [
    c for c in removed_features
    if not c.startswith("WT")
]

print("Original candidate features:", X.shape[1])
print("Current selected features:", X_all.shape[1])

print("\nRemoved survey-weight columns:", len(weight_features))
print("Other removed columns:", other_removed)

print("\nFirst 20 removed weight columns:")
print(weight_features[:20])

print("\nRemaining survey-weight columns:")
print([c for c in X_all.columns if c.startswith("WT")])


Original candidate features: 261
Current selected features: 142

Removed survey-weight columns: 108
Other removed columns: ['SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'DMDEDUC3', 'DMDSCHOL', 'SDMVPSU', 'SDMVSTRA', 'SDJ1REPN', 'DMAETHN', 'DMARACE', 'ELIGSTAT']

First 20 removed weight columns:
['WTINT2YR', 'WTINT4YR', 'WTMEC2YR', 'WTMEC4YR', 'WTMREP01', 'WTMREP02', 'WTMREP03', 'WTMREP04', 'WTMREP05', 'WTMREP06', 'WTMREP07', 'WTMREP08', 'WTMREP09', 'WTMREP10', 'WTMREP11', 'WTMREP12', 'WTMREP13', 'WTMREP14', 'WTMREP15', 'WTMREP16']

Remaining survey-weight columns:
[]


## Full-Feature Model Evaluation

Train Logistic Regression and Random Forest using 142
selected NHANES predictors.

Evaluate both models using stratified 5-fold cross-validation
and compare their performance with our original 3-feature models.

In [50]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Check the prepared data
assert X_all.shape[0] == len(y_all)
assert set(numeric_all + categorical_all) == set(X_all.columns)
assert set(numeric_all).isdisjoint(categorical_all)

# Preprocessing fitted separately inside each training fold
def make_preprocessor(scale_numeric=False):

    numeric_steps = [
        ("imputer", SimpleImputer(
            strategy="median",
            keep_empty_features=True
        ))
    ]

    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_all),

        ("categorical", Pipeline([
            ("imputer", SimpleImputer(
                strategy="most_frequent",
                keep_empty_features=True
            )),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore"
            ))
        ]), categorical_all)
    ])

# Build both models
full_models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", make_preprocessor(scale_numeric=True)),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessing", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# Same 5-fold validation
cv_all = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metrics = {
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC-AUC": "roc_auc"
}

full_model_results = []

for name, pipeline in full_models.items():

    print(f"\nTraining {name} with {X_all.shape[1]} features...")

    scores = cross_validate(
        pipeline,
        X_all,
        y_all,
        cv=cv_all,
        scoring=metrics,
        error_score="raise"
    )

    fold_results = pd.DataFrame({
        metric: scores[f"test_{metric}"]
        for metric in metrics
    })

    display(fold_results.round(3))

    full_model_results.append({
        "Model": name,
        **fold_results.mean().to_dict()
    })

# Final comparison
full_comparison = pd.DataFrame(
    full_model_results
).set_index("Model")

print("\nFull-Feature Model Comparison:")
display(full_comparison.round(3))


Training Logistic Regression with 142 features...


,Precision,Recall,F1,ROC-AUC
0,0.360,0.726,0.481,0.842
1,0.348,0.757,0.476,0.845
2,0.347,0.764,0.478,0.860
3,0.377,0.798,0.512,0.876
4,0.365,0.807,0.503,0.864



Training Random Forest with 142 features...


,Precision,Recall,F1,ROC-AUC
0,0.430,0.614,0.506,0.846
1,0.438,0.676,0.531,0.855
2,0.439,0.671,0.531,0.870
3,0.492,0.713,0.582,0.895
4,0.453,0.668,0.540,0.876



Full-Feature Model Comparison:


,Precision,Recall,F1,ROC-AUC
Model,,,,
Logistic Regression,0.359,0.770,0.490,0.857
Random Forest,0.450,0.668,0.538,0.868


### Full-Feature Experiment — Preliminary Results

We expanded Logistic Regression and Random Forest from
3 features to 142 selected NHANES predictors.

Both models improved their ROC-AUC scores:

- Logistic Regression: 0.816 to 0.857
- Random Forest: 0.833 to 0.868

Random Forest achieved the higher ROC-AUC and F1-score,
while Logistic Regression achieved higher recall.

These results use stratified 5-fold cross-validation
and predict 5-year all-cause mortality.

The 142 predictors are an exploratory feature set.
Further feature validation and outcome checks are needed.

## Comparing Full-Feature Model Predictions

Train Logistic Regression and Random Forest on the same
training participants and compare predictions on the same
held-out participants.

This experiment uses 142 preliminary NHANES predictors.

In [51]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Same participants for both models
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    stratify=y_all,
    random_state=42
)

print("Training participants:", len(X_train_all))
print("Test participants:", len(X_test_all))
print("Features:", X_train_all.shape[1])

# 2. Train Logistic Regression on training data only
glm_full_fit = clone(full_models["Logistic Regression"])
glm_full_fit.fit(X_train_all, y_train_all)

# 3. Train Random Forest on the SAME training data
rf_full_fit = clone(full_models["Random Forest"])
rf_full_fit.fit(X_train_all, y_train_all)

# 4. Get prediction scores for the SAME test participants
glm_prob_all = glm_full_fit.predict_proba(X_test_all)[:, 1]
rf_prob_all = rf_full_fit.predict_proba(X_test_all)[:, 1]

# 5. Compare predictions
prediction_comparison_all = pd.DataFrame({
    "Actual_Death": y_test_all,
    "GLM_Probability": glm_prob_all,
    "RF_Probability": rf_prob_all
}, index=X_test_all.index)

prediction_comparison_all["GLM_Predicted"] = (
    prediction_comparison_all["GLM_Probability"] >= 0.5
).astype(int)

prediction_comparison_all["RF_Predicted"] = (
    prediction_comparison_all["RF_Probability"] >= 0.5
).astype(int)

prediction_comparison_all["Difference"] = np.abs(
    prediction_comparison_all["GLM_Probability"]
    - prediction_comparison_all["RF_Probability"]
)

disagreement_all = (
    prediction_comparison_all["GLM_Predicted"]
    != prediction_comparison_all["RF_Predicted"]
)

# 6. Display results
print("\nFull-Feature Prediction Comparison")
print("---------------------------------")

print("Test participants:", len(X_test_all))
print("Different classifications:", disagreement_all.sum())
print("Disagreement rate:",
      round(disagreement_all.mean() * 100, 2), "%")

print("Average probability difference:",
      round(prediction_comparison_all["Difference"].mean(), 3))

print("\nHeld-out ROC-AUC:")
print("Logistic Regression:",
      round(roc_auc_score(y_test_all, glm_prob_all), 3))
print("Random Forest:",
      round(roc_auc_score(y_test_all, rf_prob_all), 3))

print("\nParticipants with largest differences:")
display(
    prediction_comparison_all
    .sort_values("Difference", ascending=False)
    .head(10)
    .round(3)
)

Training participants: 7965
Test participants: 1992
Features: 142

Full-Feature Prediction Comparison
---------------------------------
Test participants: 1992
Different classifications: 233
Disagreement rate: 11.7 %
Average probability difference: 0.105

Held-out ROC-AUC:
Logistic Regression: 0.842
Random Forest: 0.865

Participants with largest differences:


,Actual_Death,GLM_Probability,RF_Probability,GLM_Predicted,RF_Predicted,Difference
3540,0,0.991,0.214,1,0,0.777
1028,0,0.994,0.258,1,0,0.736
5400,0,0.894,0.219,1,0,0.674
3132,0,0.837,0.181,1,0,0.656
9157,0,0.932,0.277,1,0,0.655
2380,0,0.766,0.146,1,0,0.620
5845,1,0.000,0.600,0,1,0.600
732,0,0.879,0.303,1,0,0.576
6068,0,0.629,0.061,1,0,0.568
9440,0,0.811,0.249,1,0,0.561


## Full-Feature SHAP Explanation Comparison

Compare Logistic Regression and Random Forest explanations
for the same held-out participants using 142 NHANES predictors.

In [52]:
# Select 10 held-out participants for our first SHAP test
people_shap_all = X_test_all.sample(
    n=10,
    random_state=123
)

# Get fitted preprocessing from both models
glm_preprocessor = glm_full_fit.named_steps["preprocessing"]
rf_preprocessor = rf_full_fit.named_steps["preprocessing"]

# Transform the same participants
glm_transformed = glm_preprocessor.transform(people_shap_all)
rf_transformed = rf_preprocessor.transform(people_shap_all)

# Verify feature dimensions
glm_names = glm_preprocessor.get_feature_names_out()
rf_names = rf_preprocessor.get_feature_names_out()

print("Participants:", len(people_shap_all))
print("Original NHANES predictors:", people_shap_all.shape[1])

print("\nLogistic Regression model inputs:",
      glm_transformed.shape[1])

print("Random Forest model inputs:",
      rf_transformed.shape[1])

print("\nFeature names match:",
      np.array_equal(glm_names, rf_names))

print("\nFirst 15 encoded features:")
print(glm_names[:15])

Participants: 10
Original NHANES predictors: 142

Logistic Regression model inputs: 232
Random Forest model inputs: 232

Feature names match: True

First 15 encoded features:
['numeric__RIDAGEYR' 'numeric__RIDAGEMN' 'numeric__RIDAGEEX'
 'numeric__DMDHHSIZ' 'numeric__INDFMPIR' 'numeric__DMDHRAGE'
 'numeric__SIALANG' 'numeric__SIAPROXY' 'numeric__SIAINTRP'
 'numeric__FIALANG' 'numeric__FIAPROXY' 'numeric__FIAINTRP'
 'numeric__MIALANG' 'numeric__MIAPROXY' 'numeric__MIAINTRP']


In [55]:
import shap
import numpy as np
import pandas as pd

# Use the same three held-out participants
people_small = people_shap_all.head(3)

# Use the same background for both models
background = X_train_all.sample(n=8, random_state=42)

feature_names = X_train_all.columns.tolist()

# Convert NHANES numeric category codes to consistent float types
# Do not silently replace nonnumeric values
background_float = background[feature_names].astype("float64")
people_float = people_small[feature_names].astype("float64")

# SHAP receives numerical arrays
background_array = background_float.to_numpy()
people_array = people_float.to_numpy()

# Reconstruct the original feature columns for each model
def predict_death(model):
    def predict(data):
        df = pd.DataFrame(data, columns=feature_names)
        return model.predict_proba(df)[:, 1]
    return predict

glm_predict = predict_death(glm_full_fit)
rf_predict = predict_death(rf_full_fit)

# Verify that conversion did not change model predictions
assert np.allclose(
    glm_predict(people_array),
    glm_full_fit.predict_proba(people_small)[:, 1]
)

assert np.allclose(
    rf_predict(people_array),
    rf_full_fit.predict_proba(people_small)[:, 1]
)

# Build explainers using the same numerical background
glm_explainer_all = shap.Explainer(
    glm_predict,
    background_array,
    algorithm="permutation",
    feature_names=feature_names
)

rf_explainer_all = shap.Explainer(
    rf_predict,
    background_array,
    algorithm="permutation",
    feature_names=feature_names
)

evaluations = 2 * len(feature_names) + 1

print("Calculating Logistic Regression SHAP...")

glm_shap_small = glm_explainer_all(
    people_array,
    max_evals=evaluations
)

print("Calculating Random Forest SHAP...")

rf_shap_small = rf_explainer_all(
    people_array,
    max_evals=evaluations
)

# Find the most influential feature for each participant
glm_top = np.abs(glm_shap_small.values).argmax(axis=1)
rf_top = np.abs(rf_shap_small.values).argmax(axis=1)

comparison_small = pd.DataFrame({
    "GLM_Top_Feature": np.array(feature_names)[glm_top],
    "RF_Top_Feature": np.array(feature_names)[rf_top]
}, index=people_small.index)

comparison_small["Agree"] = (
    comparison_small["GLM_Top_Feature"]
    == comparison_small["RF_Top_Feature"]
)

print("\nFull-Feature SHAP Comparison:")
display(comparison_small)

print(
    "\nTop-feature agreement:",
    round(comparison_small["Agree"].mean() * 100, 1),
    "%"
)

Calculating Logistic Regression SHAP...
Calculating Random Forest SHAP...

Full-Feature SHAP Comparison:


,GLM_Top_Feature,RF_Top_Feature,Agree
4764,LBXSNASI,RIDAGEYR,False
3217,LBXSGB,RIDAGEYR,False
5260,LBXLYPCT,RIDAGEYR,False



Top-feature agreement: 0.0 %


In [56]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# Select 30 held-out participants
people_30 = X_test_all.sample(
    n=30,
    random_state=123
)

people_array_30 = (
    people_30[feature_names]
    .astype("float64")
    .to_numpy()
)

# Explain the same people using both models
print("Calculating Logistic Regression SHAP...")
glm_shap_30 = glm_explainer_all(
    people_array_30,
    max_evals=evaluations
)

print("Calculating Random Forest SHAP...")
rf_shap_30 = rf_explainer_all(
    people_array_30,
    max_evals=evaluations
)

# Compare absolute SHAP feature importance
glm_abs = np.abs(glm_shap_30.values)
rf_abs = np.abs(rf_shap_30.values)

# Most important feature for each participant
glm_top = glm_abs.argmax(axis=1)
rf_top = rf_abs.argmax(axis=1)

top_agree = glm_top == rf_top

# Full feature rankings
glm_ranks = np.argsort(-glm_abs, axis=1)
rf_ranks = np.argsort(-rf_abs, axis=1)

full_agree = np.all(glm_ranks == rf_ranks, axis=1)

# Correlation between feature-importance rankings
correlations = [
    spearmanr(glm_abs[i], rf_abs[i]).statistic
    for i in range(len(people_30))
]

# Results
print("\n142-Feature SHAP Agreement")
print("--------------------------")

print("Participants:", len(people_30))
print("Top-feature agreement:",
      round(top_agree.mean() * 100, 1), "%")

print("Complete ranking agreement:",
      round(full_agree.mean() * 100, 1), "%")

print("Average Spearman correlation:",
      round(np.nanmean(correlations), 3))

# Show individual comparisons
comparison_30 = pd.DataFrame({
    "GLM_Top": np.array(feature_names)[glm_top],
    "RF_Top": np.array(feature_names)[rf_top],
    "Agree": top_agree,
    "Spearman": correlations
}, index=people_30.index)

display(comparison_30.head(10).round(3))

Calculating Logistic Regression SHAP...
Calculating Random Forest SHAP...

142-Feature SHAP Agreement
--------------------------
Participants: 30
Top-feature agreement: 6.7 %
Complete ranking agreement: 0.0 %
Average Spearman correlation: 0.55


,GLM_Top,RF_Top,Agree,Spearman
4764,LBXSNASI,RIDAGEYR,False,0.501
3217,LBXNEPCT,RIDAGEYR,False,0.528
5260,LBXLYPCT,RIDAGEYR,False,0.511
3179,LBXNEPCT,RIDAGEYR,False,0.443
2171,LBXNEPCT,SSBNP,False,0.559
9223,LBXSGL,RIDAGEMN,False,0.546
4924,LBXNEPCT,SSBNP,False,0.565
3759,LBXSOSSI,LBXBCD,False,0.540
1930,LBXNEPCT,SSBNP,False,0.617
5298,LBXNEPCT,SSBNP,False,0.566


In [57]:
# Compare top-5 and top-10 SHAP features for each person

def top_k_overlap(glm_values, rf_values, k):
    overlaps = []

    for i in range(len(glm_values)):
        glm_top = set(np.argsort(-glm_values[i])[:k])
        rf_top = set(np.argsort(-rf_values[i])[:k])

        shared = len(glm_top & rf_top)
        overlaps.append(shared / k)

    return np.array(overlaps)


top5_overlap = top_k_overlap(glm_abs, rf_abs, 5)
top10_overlap = top_k_overlap(glm_abs, rf_abs, 10)

print("Participants:", len(people_30))

print(
    "Average top-5 overlap:",
    round(top5_overlap.mean() * 100, 1), "%"
)

print(
    "Average top-10 overlap:",
    round(top10_overlap.mean() * 100, 1), "%"
)

print(
    "Participants sharing at least one top-5 feature:",
    int((top5_overlap > 0).sum())
)

# Show participant-level results
overlap_comparison = comparison_30.copy()

overlap_comparison["Top5_Overlap"] = top5_overlap
overlap_comparison["Top10_Overlap"] = top10_overlap

display(overlap_comparison.head(10).round(3))

Participants: 30
Average top-5 overlap: 8.0 %
Average top-10 overlap: 10.7 %
Participants sharing at least one top-5 feature: 12


,GLM_Top,RF_Top,Agree,Spearman,Top5_Overlap,Top10_Overlap
4764,LBXSNASI,RIDAGEYR,False,0.501,0.0,0.0
3217,LBXNEPCT,RIDAGEYR,False,0.528,0.0,0.1
5260,LBXLYPCT,RIDAGEYR,False,0.511,0.0,0.1
3179,LBXNEPCT,RIDAGEYR,False,0.443,0.2,0.1
2171,LBXNEPCT,SSBNP,False,0.559,0.0,0.1
9223,LBXSGL,RIDAGEMN,False,0.546,0.2,0.3
4924,LBXNEPCT,SSBNP,False,0.565,0.0,0.0
3759,LBXSOSSI,LBXBCD,False,0.540,0.0,0.0
1930,LBXNEPCT,SSBNP,False,0.617,0.0,0.1
5298,LBXNEPCT,SSBNP,False,0.566,0.2,0.2


In [58]:
# Check SHAP stability using a different background sample

background_40 = (
    X_train_all[feature_names]
    .sample(n=40, random_state=99)
    .astype("float64")
    .to_numpy()
)

glm_explainer_40 = shap.Explainer(
    glm_predict,
    background_40,
    algorithm="permutation",
    feature_names=feature_names
)

rf_explainer_40 = shap.Explainer(
    rf_predict,
    background_40,
    algorithm="permutation",
    feature_names=feature_names
)

# Explain the same 30 participants
glm_shap_40 = glm_explainer_40(
    people_array_30,
    max_evals=evaluations
)

rf_shap_40 = rf_explainer_40(
    people_array_30,
    max_evals=evaluations
)

# Top features with the new background
glm_top_40 = np.abs(glm_shap_40.values).argmax(axis=1)
rf_top_40 = np.abs(rf_shap_40.values).argmax(axis=1)

print("Participants:", len(people_30))

print(
    "New top-feature agreement:",
    round((glm_top_40 == rf_top_40).mean() * 100, 1),
    "%"
)

print(
    "GLM explanation stability:",
    round((glm_top == glm_top_40).mean() * 100, 1),
    "%"
)

print(
    "Random Forest explanation stability:",
    round((rf_top == rf_top_40).mean() * 100, 1),
    "%"
)

Participants: 30
New top-feature agreement: 3.3 %
GLM explanation stability: 36.7 %
Random Forest explanation stability: 63.3 %


In [59]:
# Compare two different SHAP backgrounds of the same size

background_A = X_train_all[feature_names].sample(
    n=40, random_state=42
).astype("float64").to_numpy()

background_B = X_train_all[feature_names].sample(
    n=40, random_state=99
).astype("float64").to_numpy()

def test_background(model_predict, background):
    explainer = shap.Explainer(
        model_predict,
        background,
        algorithm="permutation",
        feature_names=feature_names,
        seed=42
    )

    return explainer(
        people_array_30,
        max_evals=evaluations
    )

# Explain the same 30 participants
glm_A = test_background(glm_predict, background_A)
glm_B = test_background(glm_predict, background_B)

rf_A = test_background(rf_predict, background_A)
rf_B = test_background(rf_predict, background_B)

# Identify each participant's most important feature
glm_top_A = np.abs(glm_A.values).argmax(axis=1)
glm_top_B = np.abs(glm_B.values).argmax(axis=1)

rf_top_A = np.abs(rf_A.values).argmax(axis=1)
rf_top_B = np.abs(rf_B.values).argmax(axis=1)

print("Logistic Regression stability:",
      round((glm_top_A == glm_top_B).mean() * 100, 1), "%")

print("Random Forest stability:",
      round((rf_top_A == rf_top_B).mean() * 100, 1), "%")

print("Model agreement with background A:",
      round((glm_top_A == rf_top_A).mean() * 100, 1), "%")

print("Model agreement with background B:",
      round((glm_top_B == rf_top_B).mean() * 100, 1), "%")

Logistic Regression stability: 50.0 %
Random Forest stability: 73.3 %
Model agreement with background A: 6.7 %
Model agreement with background B: 10.0 %


In [60]:
# Compare two different SHAP backgrounds of the same size

background_A = X_train_all[feature_names].sample(
    n=40, random_state=42
).astype("float64").to_numpy()

background_B = X_train_all[feature_names].sample(
    n=40, random_state=99
).astype("float64").to_numpy()

def test_background(model_predict, background):
    explainer = shap.Explainer(
        model_predict,
        background,
        algorithm="permutation",
        feature_names=feature_names,
        seed=42
    )

    return explainer(
        people_array_30,
        max_evals=evaluations
    )

# Explain the same 30 participants
glm_A = test_background(glm_predict, background_A)
glm_B = test_background(glm_predict, background_B)

rf_A = test_background(rf_predict, background_A)
rf_B = test_background(rf_predict, background_B)

# Identify each participant's most important feature
glm_top_A = np.abs(glm_A.values).argmax(axis=1)
glm_top_B = np.abs(glm_B.values).argmax(axis=1)

rf_top_A = np.abs(rf_A.values).argmax(axis=1)
rf_top_B = np.abs(rf_B.values).argmax(axis=1)

print("Logistic Regression stability:",
      round((glm_top_A == glm_top_B).mean() * 100, 1), "%")

print("Random Forest stability:",
      round((rf_top_A == rf_top_B).mean() * 100, 1), "%")

print("Model agreement with background A:",
      round((glm_top_A == rf_top_A).mean() * 100, 1), "%")

print("Model agreement with background B:",
      round((glm_top_B == rf_top_B).mean() * 100, 1), "%")

Logistic Regression stability: 50.0 %
Random Forest stability: 73.3 %
Model agreement with background A: 6.7 %
Model agreement with background B: 10.0 %


In [61]:
# Check top-5 explanation stability
# Reuse the SHAP results we already calculated

def top5_overlap(values_a, values_b):
    top_a = np.argsort(-np.abs(values_a), axis=1)[:, :5]
    top_b = np.argsort(-np.abs(values_b), axis=1)[:, :5]

    overlaps = []

    for a, b in zip(top_a, top_b):
        shared = len(set(a) & set(b))
        overlaps.append(shared / 5)

    return np.array(overlaps)


# Stability within each model
glm_stability_5 = top5_overlap(
    glm_A.values, glm_B.values
)

rf_stability_5 = top5_overlap(
    rf_A.values, rf_B.values
)

# Agreement between the two models
agreement_5_A = top5_overlap(
    glm_A.values, rf_A.values
)

agreement_5_B = top5_overlap(
    glm_B.values, rf_B.values
)

print("Top-5 Feature Comparison")
print("------------------------")

print("Logistic Regression stability:",
      round(glm_stability_5.mean() * 100, 1), "%")

print("Random Forest stability:",
      round(rf_stability_5.mean() * 100, 1), "%")

print("Between-model agreement, background A:",
      round(agreement_5_A.mean() * 100, 1), "%")

print("Between-model agreement, background B:",
      round(agreement_5_B.mean() * 100, 1), "%")

Top-5 Feature Comparison
------------------------
Logistic Regression stability: 76.0 %
Random Forest stability: 80.0 %
Between-model agreement, background A: 7.3 %
Between-model agreement, background B: 9.3 %
